In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading Full Dataset for Cross Validation...")
# Note: In K-fold, we generally merge train and val (or just use train) to do the folds.
# Here we will use the combined X_train and X_val.
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_val = pd.read_csv('../data/processed/y_val.csv').values.ravel()

X_cv = pd.concat([X_train, X_val], ignore_index=True)
y_cv = np.concatenate([y_train, y_val])
print(f"CV Dataset Shape: {X_cv.shape}")

In [ ]:
print("Running 5-Fold Stratified Cross Validation with XGBoost...")
scale_pos_weight = (len(y_cv) - sum(y_cv)) / sum(y_cv)
model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {'roc_auc': 'roc_auc', 'f1': 'f1', 'recall': 'recall', 'precision': 'precision'}
cv_results = cross_validate(model, X_cv, y_cv, cv=skf, scoring=scoring, n_jobs=-1, return_train_score=False)

In [ ]:
metrics = ['test_roc_auc', 'test_f1', 'test_recall', 'test_precision']
display_names = ['ROC-AUC', 'F1-Score', 'Recall', 'Precision']

cv_summary = {}
print("\n--- 5-Fold CV Results ---")
for metric, name in zip(metrics, display_names):
    mean_val = cv_results[metric].mean()
    std_val = cv_results[metric].std()
    cv_summary[name] = {'mean': float(mean_val), 'std': float(std_val)}
    print(f"{name}: {mean_val:.4f} (+/- {std_val*2:.4f})")

os.makedirs('../models', exist_ok=True)
with open('../models/cross_validation_results.json', 'w') as f:
    json.dump(cv_summary, f, indent=4)

# Visualization
plt.figure(figsize=(10, 6))
sns.boxplot(data=pd.DataFrame({name: cv_results[metric] for metric, name in zip(metrics, display_names)}))
plt.title('5-Fold Cross Validation Metrics Distribution')
plt.ylabel('Score')
os.makedirs('../visualizations/evaluation', exist_ok=True)
plt.savefig('../visualizations/evaluation/cv_metrics_boxplot.png')
plt.close()
print("Saved cv_metrics_boxplot.png")